In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Test script for two ontology‑based tools:
1. Price prediction (using saved ensemble model)
2. Sentiment analysis (ontology‑adjusted score for a headline)

Requires the saved models to be in /Users/christang/Downloads/models/
"""

import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from pathlib import Path
from textblob import TextBlob
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ----------------------------------------------------------------------
# 1. Paths and model loading
# ----------------------------------------------------------------------
MODEL_DIR = Path("/Users/christang/Downloads/models")

class ModelLoader:
    """Load all saved models and scalers once."""
    def __init__(self):
        self.gbm = joblib.load(MODEL_DIR / "gbm_model.pkl")
        self.lstm = tf.keras.models.load_model(MODEL_DIR / "lstm_model.keras")
        self.meta = joblib.load(MODEL_DIR / "meta_model.pkl")
        self.price_scaler = joblib.load(MODEL_DIR / "global_price_scaler.pkl")
        self.sent_scaler = joblib.load(MODEL_DIR / "global_sent_scaler.pkl")
        self.metadata = joblib.load(MODEL_DIR / "metadata.pkl")
        self.lookback = self.metadata['lookback']
        self.threshold = self.metadata['threshold']

    def predict_from_features(self, price_df, sentiment_df):
        """
        price_df: DataFrame with 20 rows, columns = price_cols (10 features)
        sentiment_df: DataFrame with 20 rows, columns = sentiment_cols (8 features)
        Returns (probability_up, signal)
        """
        # Scale
        price_scaled = self.price_scaler.transform(price_df.values)
        sent_scaled = self.sent_scaler.transform(sentiment_df.values)

        # LSTM input
        price_seq = price_scaled.reshape(1, self.lookback, -1)
        sent_seq = sent_scaled.reshape(1, self.lookback, -1)
        prob_lstm = self.lstm.predict([price_seq, sent_seq], verbose=0)[0][0]

        # GBM input (flattened)
        flat = np.concatenate([
            price_scaled.mean(axis=0), price_scaled.max(axis=0), price_scaled.std(axis=0),
            sent_scaled.mean(axis=0), sent_scaled.max(axis=0), sent_scaled.std(axis=0),
            price_scaled[-1], sent_scaled[-1]
        ]).reshape(1, -1)
        prob_gbm = self.gbm.predict_proba(flat)[0][1]

        # Stacking
        prob = self.meta.predict_proba(np.array([[prob_lstm, prob_gbm]]))[0][1]
        signal = "BUY" if prob > self.threshold else "SELL"
        return prob, signal

# ----------------------------------------------------------------------
# 2. Sentiment analysis (ontology‑adjusted)
# ----------------------------------------------------------------------
# Minimal entity map for demonstration (replace with your full map from training)
ENTITY_MAP = {
    "700": {"Alibaba": "competitor", "HSI": "index", "Southbound": "institution"},
    "1810": {"Apple": "competitor", "Qualcomm": "supplier", "TSMC": "supplier"},
    "9988": {"Tencent": "competitor"},
    # Add more as needed
}

class OntologySentimentAnalyzer:
    def __init__(self):
        self.model = None
        self.pos_words = {"surge","soar","rally","gain","upgrade","beat","exceed",
                          "growth","profit","bullish","buy","strong","opportunity",
                          "breakthrough","positive","record","high","rise","increase",
                          "outperform"}
        self.neg_words = {"drop","fall","decline","downgrade","miss","below","loss",
                          "bearish","sell","weak","risk","warning","negative","low",
                          "decrease","plunge","crash","concern"}
        try:
            print("Loading FinBERT...")
            self.tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
            self.model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.model.to(self.device)
            self.model.eval()
            print("FinBERT loaded.")
        except Exception as e:
            print(f"FinBERT failed: {e}. Using lexicon fallback.")
            self.model = None

    def get_raw_sentiment(self, text):
        if self.model is not None:
            try:
                inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                with torch.no_grad():
                    probs = torch.nn.functional.softmax(self.model(**inputs).logits, dim=-1)
                    probs = probs.cpu().numpy()[0]
                return probs[2] - probs[0]
            except:
                pass
        # Lexicon fallback
        text_lower = text.lower()
        pos_cnt = sum(1 for w in self.pos_words if w in text_lower)
        neg_cnt = sum(1 for w in self.neg_words if w in text_lower)
        total = pos_cnt + neg_cnt
        if total == 0:
            return TextBlob(text).sentiment.polarity
        else:
            return (pos_cnt - neg_cnt) / total

    def get_ontology_sentiment(self, title, ticker):
        raw = self.get_raw_sentiment(title)
        if ticker in ENTITY_MAP:
            for ent, rel in ENTITY_MAP[ticker].items():
                if ent.lower() in title.lower():
                    if rel == "competitor":
                        return -raw
                    else:
                        return raw   # match relation (supplier, index, etc.)
        return raw

# ----------------------------------------------------------------------
# 3. Test the two functions with dummy data
# ----------------------------------------------------------------------
def test_prediction():
    print("\n" + "="*60)
    print("Testing price prediction with dummy 20‑day data...")
    print("="*60)

    # Load models
    loader = ModelLoader()
    price_cols = loader.metadata['price_cols']
    sent_cols = loader.metadata['sentiment_cols']

    # Generate dummy 20‑day data (random numbers)
    np.random.seed(42)
    dummy_price = pd.DataFrame(np.random.randn(20, len(price_cols)), columns=price_cols)
    dummy_sent = pd.DataFrame(np.random.randn(20, len(sent_cols)), columns=sent_cols)

    # For sentiment columns, ensure they are within realistic ranges (e.g., positive/negative)
    dummy_sent = dummy_sent.abs()  # make all positive (not realistic but just for test)
    dummy_sent.iloc[:, 0] = dummy_sent.iloc[:, 0] * 0.2  # sentiment_mean between 0 and 0.2

    prob, signal = loader.predict_from_features(dummy_price, dummy_sent)
    print(f"Dummy prediction -> Probability up: {prob:.4f}, Signal: {signal}")
    print("Prediction test completed.\n")

def test_sentiment_analysis():
    print("\n" + "="*60)
    print("Testing ontology‑adjusted sentiment analysis...")
    print("="*60)

    analyzer = OntologySentimentAnalyzer()

    # Example 1: Competitor news
    title = "Apple announces record iPhone sales, Samsung stock falls"
    ticker = "1810"   # Xiaomi (competitor of Apple)
    score = analyzer.get_ontology_sentiment(title, ticker)
    print(f"Title: {title}")
    print(f"Ticker: {ticker} -> Ontology sentiment: {score:.4f} (should be negative because competitor positive news is inverted)\n")

    # Example 2: Supplier news (should keep raw sentiment)
    title = "Qualcomm unveils new 5G chip, boosting smartphone performance"
    ticker = "1810"   # Xiaomi uses Qualcomm chips
    score = analyzer.get_ontology_sentiment(title, ticker)
    print(f"Title: {title}")
    print(f"Ticker: {ticker} -> Ontology sentiment: {score:.4f} (should be positive, no inversion)\n")

    # Example 3: No relation
    title = "Pandemic fears drive market selloff"
    ticker = "700"    # Tencent, no clear relation
    score = analyzer.get_ontology_sentiment(title, ticker)
    print(f"Title: {title}")
    print(f"Ticker: {ticker} -> Ontology sentiment: {score:.4f} (raw, no change)\n")

# ----------------------------------------------------------------------
# Run tests
# ----------------------------------------------------------------------
if __name__ == "__main__":
    test_prediction()
    test_sentiment_analysis()
    print("All tests completed.")

/Users/christang/miniconda3/envs/comp3340/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Testing price prediction with dummy 20‑day data...


/Users/christang/miniconda3/envs/comp3340/lib/python3.10/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (1, 20, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


Dummy prediction -> Probability up: 0.5400, Signal: BUY
Prediction test completed.


Testing ontology‑adjusted sentiment analysis...
Loading FinBERT...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 25601.43it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT loaded.
Title: Apple announces record iPhone sales, Samsung stock falls
Ticker: 1810 -> Ontology sentiment: -0.4765 (should be negative because competitor positive news is inverted)

Title: Qualcomm unveils new 5G chip, boosting smartphone performance
Ticker: 1810 -> Ontology sentiment: -0.9045 (should be positive, no inversion)

Title: Pandemic fears drive market selloff
Ticker: 700 -> Ontology sentiment: 0.0399 (raw, no change)

All tests completed.
